In [1]:
# 🔥 CLEAN INSTALL (ONLY THIS)

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install torchvision

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-6kbjm3ot/unsloth_8fd71670ef544963b1c7734781d7ec14
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-6kbjm3ot/unsloth_8fd71670ef544963b1c7734781d7ec14
  Resolved https://github.com/unslothai/unsloth.git to commit b09aa82a3ac9ac9794c497e4b8f747f77e52b162
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.0 MB/s eta 0:00:0

In [1]:
import unsloth
print("Unsloth working ✅")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth working ✅


In [2]:
import unsloth
from unsloth import FastLanguageModel

import torch, random, json, os, inspect
from datetime import datetime
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# 🔹 Config
MODEL_ID = "unsloth/Qwen2.5-1.5B-bnb-4bit"
MAX_SEQ_LENGTH = 512
DIFFICULTY = "medium"
WANDB_PROJECT = None

In [3]:
DOMAINS = [
    'data_pipeline','api_workflow','report_generation','system_config',
    'multi_agent_coordination','code_review','incident_response',
    'personal_assistant','large_scale_migration'
]

CONTEXTS = [
    'A new enterprise task has been assigned.',
    'This is a critical production workflow.',
    'Handle carefully — failures have business impact.',
    'Security and compliance policies are active.',
    'Long-running task — checkpoint state regularly.',
]

FAILURES = [
    "api_500","rate_limit_429","auth_401",
    "timeout","cascading_failure","security_breach"
]

def generate_sample(i):
    domain = DOMAINS[i % len(DOMAINS)]
    context = CONTEXTS[i % len(CONTEXTS)]
    failure = random.choice(FAILURES)

    return {
        "prompt": [
            {"role": "system", "content": "You are an AI agent."},
            {"role": "user", "content": (
                f"You are in a {DIFFICULTY} production environment. "
                f"{context} Domain: {domain}. "
                f"Failure occurred: {failure}. "
                "Respond in JSON."
            )}
        ]
    }

DATASET_SIZE = 200
dataset = Dataset.from_list([generate_sample(i) for i in range(DATASET_SIZE)])

print("Dataset ready:", len(dataset))

Dataset ready: 200


In [10]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# 🔥 FIX HERE
tokenizer.chat_template = "{% for message in messages %}{{ message['role'] + ': ' + message['content'] + '\n' }}{% endfor %}"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj','k_proj','v_proj','o_proj',
        'gate_proj','up_proj','down_proj'
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/Qwen2.5-1.5B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [11]:
class AgentAction:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

class DummyObs:
    def __init__(self):
        self._reward = random.uniform(0, 5)
        self.is_done = True

class Env:
    def reset(self):
        return DummyObs()
    def step(self, action):
        return DummyObs()

env = Env()

In [12]:
def parse_action(text):
    try:
        s = text.find("{")
        e = text.rfind("}") + 1
        return AgentAction(**json.loads(text[s:e]))
    except:
        return AgentAction(action_type="invalid")

def reward_fn(completions, **kwargs):
    rewards = []

    for c in completions:
        text = str(c)

        obs = env.reset()
        action = parse_action(text)
        obs = env.step(action)

        reward = obs._reward

        if "detect_failure" in text:
            reward += 2
        if "recover" in text:
            reward += 3

        rewards.append(float(reward))

    return rewards

print("Reward ready ✅")

Reward ready ✅


In [13]:
OUTPUT_DIR = f"./outputs/{datetime.now().strftime('%H%M%S')}"

config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_completion_length=200,
    logging_steps=1,
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


In [14]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    train_dataset=dataset,
    args=config,
)

print("Trainer ready 🚀")

Trainer ready 🚀


In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` wil

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_fn / mean,rewards / reward_fn / std
1,-0.049225,2.323601,1.583171,121.437500,13.000000,200.000000,0.343750,80.285713,13.000000,198.000000,-0.000000,2.323601,1.643753
2,-0.070315,2.342568,1.224588,162.218750,16.000000,200.000000,0.593750,107.000008,16.000000,185.000000,-0.000000,2.342568,1.317127
3,-0.036800,3.003783,1.280788,152.625000,28.000000,200.000000,0.531250,98.933342,28.000000,186.000000,0.000012,3.003783,1.248196
4,-0.069915,2.425057,1.571940,146.968750,38.000000,200.000000,0.468750,100.176468,38.000000,200.000000,0.000012,2.425057,1.616608
5,0.019063,2.579421,1.528131,142.593750,10.000000,200.000000,0.437500,97.944443,10.000000,193.000000,0.000011,2.579421,1.659748
6,-0.071942,3.145363,1.996417,141.062500,22.000000,200.000000,0.500000,82.125000,22.000000,165.000000,0.000014,3.145363,1.965108
7,0.001653,2.231398,1.223259,149.750000,36.000000,200.000000,0.500000,99.500000,36.000000,199.000000,0.000012,2.231398,1.357616
8,0.003580,2.892660,1.349457,142.750000,13.000000,200.000000,0.500000,85.500000,13.000000,199.000000,0.000015,2.892660,1.342868
9,-0.058664,2.691138,1.522963,137.656250,10.000000,200.000000,0.468750,82.647057,10.000000,195.000000,0.000016,2.691138,1.550770
10,-0.090818,2.307866,1.513759,143.218750,27.000000,200.000000,0.437500,99.055557,27.000000,175.000000,0.000014,2.307866,1.554358


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

TrainOutput(global_step=50, training_loss=-0.013143202988430858, metrics={'train_runtime': 2895.4428, 'train_samples_per_second': 0.069, 'train_steps_per_second': 0.017, 'total_flos': 0.0, 'train_loss': -0.013143202988430858})

In [16]:
NUM_AGENTS = 3

agents = []
for i in range(NUM_AGENTS):
    m, t = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    m = FastLanguageModel.get_peft_model(
        m,
        r=16,
        target_modules=[
            'q_proj','k_proj','v_proj','o_proj',
            'gate_proj','up_proj','down_proj'
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias='none',
        use_gradient_checkpointing='unsloth',
    )
    agents.append(m)

print(f"{len(agents)} agents initialized 🔥")

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/Qwen2.5-1.5B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/Qwen2.5-1.5B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/Qwen2.5-1.5B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
3 agents initialized 🔥


In [24]:
def reward_fn(completions, **kwargs):
    rewards = []

    for c in completions:
        text = str(c)

        reward = 0

        if "detect_failure" in text:
            reward += 3
        if "recover" in text:
            reward += 5
        if "error" in text:
            reward -= 2

        # reasoning quality
        reward += min(len(text)/100, 2)

        rewards.append(float(reward))

    return rewards